# 1주차 · 장비 데이터 탐색
데이터: UCI SECOM (실제 웨이퍼 팹 계측 데이터)

---

## 오늘의 상황

여러분은 반도체 팹의 장비 데이터 분석 담당자입니다. 생산 라인 장비들이 웨이퍼 한 장을
처리할 때마다 590개의 센서 신호를 남기고, 그 웨이퍼는 최종 검사에서 양품(PASS) 또는
불량(FAIL) 판정을 받습니다.

공정팀에서 이런 요청이 왔습니다.

> 불량이 나는 웨이퍼는 장비 신호가 뭔가 다를 겁니다. 어떤 신호가 문제인지 찾아주세요.

오늘은 이 요청에 답하기 위해 데이터를 열어보고, 쓸 수 있는 상태로 정리하고,
불량과 관련이 있어 보이는 신호를 추려내는 것까지 합니다.
모델을 만드는 건 다음 주에 합니다.

## 규칙

`# TODO` 가 붙은 셀만 작성하면 됩니다. 각 미션 뒤에 자가진단 셀이 있으니
`[통과]` 가 뜰 때까지 스스로 고쳐보세요. 막히면 물어보되 먼저 15분은 혼자 붙들어 보세요.
정답을 베끼는 것보다 틀리고 고치는 과정이 점수에 반영됩니다.

---

## 준비

In [ ]:
# 이 셀은 그대로 실행하세요.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path

_have = {f.name for f in fm.fontManager.ttflist}
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']:
    if _f in _have:
        plt.rcParams['font.family'] = _f
        break
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 30)

DATA = Path('../../data/secom')   # 폴더를 옮겼다면 이 줄만 고치세요

def check(name, cond, hint=''):
    if cond:
        print('[통과] ' + name)
    else:
        print('[실패] ' + name + (('  ->  ' + str(hint)) if hint else ''))

print('폰트:', plt.rcParams['font.family'][0], '| 데이터 폴더:', DATA.exists())

---

## 미션 1 · 데이터 열어보기

파일이 두 개 있습니다.

- `secom_equipment.csv` — 웨이퍼 1,567장과 신호 590개, 그리고 양불 판정
- `signal_metadata.csv` — 각 신호가 어느 장비 모듈의 무슨 센서인지

두 파일을 `df`, `meta` 로 불러오고, 크기와 불량 비율을 확인하세요.
`label` 컬럼은 1이 불량, 0이 양품입니다.

`timestamp` 는 날짜로 읽어야 합니다. 여기서 `parse_dates` 를 빠뜨리면
미션 7에서 반드시 막히니 지금 챙기세요.

In [ ]:
# TODO 1-1: 두 파일 불러오기
#   힌트: pd.read_csv(DATA / '파일명', encoding='utf-8-sig', parse_dates=['timestamp'])
df = None
meta = None

# TODO 1-2: 두 데이터의 크기 출력


# TODO 1-3: 불량 건수와 비율(%)
n_fail = None
fail_rate = None

자가진단 1 — 실행해서 모두 `[통과]` 인지 확인하세요.

In [ ]:
check('데이터 로드', df is not None and meta is not None, '두 파일을 읽어오세요')
check('df 크기 1567 x 594', df is not None and df.shape == (1567, 594), None if df is None else df.shape)
check('meta 590행', meta is not None and len(meta) == 590)
check('불량 104건', n_fail == 104, n_fail)
check('불량률 6.64%', fail_rate is not None and abs(fail_rate - 6.64) < 0.05, fail_rate)

---

## 미션 2 · 결측 진단

실제 장비 데이터에는 비어 있는 값이 많습니다. 센서가 고장 났거나, 그 레시피에서는
아예 측정하지 않는 항목이거나, 통신이 끊긴 경우입니다.

신호별로 얼마나 비어 있는지 재고, 절반 넘게 비어 있는 신호를 추려내세요.
절반 이상 비어 있으면 채워 넣어도 대부분이 지어낸 값이 되기 때문에 버리는 편이 낫습니다.

In [ ]:
# TODO 2-1: SIG_ 로 시작하는 컬럼 이름만 모으기
sig_cols = None

# TODO 2-2: 신호별 결측 비율 (힌트: .isna().mean())
miss = None

# TODO 2-3: 결측이 많은 상위 10개 출력


# TODO 2-4: 결측 50%를 넘는 신호 이름 리스트
high_missing = None

자가진단 2 — 실행해서 모두 `[통과]` 인지 확인하세요.

In [ ]:
check('신호 590개', sig_cols is not None and len(sig_cols) == 590, None if sig_cols is None else len(sig_cols))
check('결측 비율 계산', miss is not None and abs(miss.mean() - 0.0454) < 0.001)
check('50% 초과 28개', high_missing is not None and len(high_missing) == 28, None if high_missing is None else len(high_missing))

---

## 미션 3 · 쓸모없는 신호 걸러내기

값이 처음부터 끝까지 똑같은 신호가 있습니다. 1,567장 내내 항상 100.0 인 식입니다.
이런 신호는 양품이든 불량이든 구별해 주지 못하니 빼고 갑니다.

미션 2에서 추린 것과 합쳐 제거 목록을 만드세요.
두 리스트를 그냥 이어 붙이지 말고 집합으로 합쳐야 합니다. 겹치는 신호가 있을 수 있으니까요.

In [ ]:
# TODO 3-1: 값이 항상 같은 신호 찾기 (힌트: .nunique())
const_cols = None

# TODO 3-2: 제거 목록 합치기 (중복 없이)
drop_cols = None

# TODO 3-3: 남는 신호
keep_cols = None

자가진단 3 — 실행해서 모두 `[통과]` 인지 확인하세요.

In [ ]:
check('상수 신호 116개', const_cols is not None and len(const_cols) == 116, None if const_cols is None else len(const_cols))
check('제거 144개', drop_cols is not None and len(drop_cols) == 144, '결측 28 + 상수 116')
check('사용 446개', keep_cols is not None and len(keep_cols) == 446, None if keep_cols is None else len(keep_cols))

---

## 미션 4 · 남은 결측 채우기

남은 446개 신호에도 결측이 조금씩 있습니다. 이건 버리기 아까우니 채웁니다.

무난한 방법은 그 신호의 중앙값으로 채우는 것입니다. 평균이 아니라 중앙값을 쓰는 이유는
센서 값에 크게 튀는 값이 섞여 있어도 덜 흔들리기 때문입니다.

In [ ]:
# TODO 4-1: 중앙값으로 채우기
#   힌트: X = df[keep_cols].fillna( ... .median())
X = None

# TODO 4-2: 남은 결측 개수와 X 크기 출력


자가진단 4 — 실행해서 모두 `[통과]` 인지 확인하세요.

In [ ]:
check('X 생성', X is not None)
check('X 크기 1567 x 446', X is not None and X.shape == (1567, 446), None if X is None else X.shape)
check('결측 0개', X is not None and int(X.isna().sum().sum()) == 0)

---

## 미션 5 · 장비 모듈별로 나눠보기

`meta` 에는 각 신호가 어느 장비 모듈의 무슨 센서인지 적혀 있습니다.
모듈은 챔버, RF전원, 진공, 가스공급, 정전척, 온도제어, 이송, 계측 여덟 가지입니다.

전처리 후 살아남은 신호가 모듈별로 몇 개씩인지 세고 막대그래프로 그리세요.

한 가지 알아둘 것이 있습니다. 이 모듈·센서 이름은 수업용으로 붙인 가상의 이름입니다.
SECOM 원본은 신호가 익명화되어 있어서 실제로 어떤 센서인지는 공개되지 않았습니다.
미션 8에서 이 점을 다시 다룹니다.

In [ ]:
# TODO 5-1: 남은 신호의 메타데이터만 추출 (힌트: .isin())
meta_keep = None

# TODO 5-2: 모듈별 개수 세기 (힌트: value_counts)
by_module = None

# TODO 5-3: 막대그래프. 제목과 축 이름을 꼭 넣으세요


자가진단 5 — 실행해서 모두 `[통과]` 인지 확인하세요.

In [ ]:
check('meta_keep 446행', meta_keep is not None and len(meta_keep) == 446, None if meta_keep is None else len(meta_keep))
check('모듈 8종', by_module is not None and len(by_module) == 8)

---

## 미션 6 · 불량 관련 신호 Top 10 찾기

오늘의 본론입니다.

불량 웨이퍼와 양품 웨이퍼의 신호 값을 비교해서 차이가 큰 신호를 찾습니다.
그런데 신호마다 단위가 다릅니다. 어떤 건 3000 언저리이고 어떤 건 0.1 언저리입니다.
그냥 빼서 비교하면 값이 큰 신호가 무조건 이깁니다.

그래서 표준편차로 나눠줍니다. 이렇게 하면 단위가 사라져서 신호끼리 공정하게 비교됩니다.
이 값을 효과크기라고 부릅니다.

```
효과크기 = | (불량 평균 - 양품 평균) / 전체 표준편차 |
```

표준편차가 0인 신호는 0으로 나누게 되니 `np.nan` 으로 바꿔두고 계산하세요.

In [ ]:
# TODO 6-1: 불량 마스크
is_fail = None

# TODO 6-2: 그룹별 평균
mean_fail = None
mean_pass = None

# TODO 6-3: 효과크기 (절댓값, 큰 순서로 정렬)
effect = None

# TODO 6-4: Top 10 을 meta 와 합쳐 표로 출력
top10 = None


# TODO 6-5: 1등 신호의 양품 vs 불량 박스플롯


자가진단 6 — 실행해서 모두 `[통과]` 인지 확인하세요.

In [ ]:
check('불량 104건', is_fail is not None and int(is_fail.sum()) == 104)
check('효과크기 계산', effect is not None and len(effect.dropna()) > 400)
check('1위 SIG_060', top10 is not None and top10.index[0] == 'SIG_060', None if top10 is None else top10.index[0])
check('2위 SIG_104', top10 is not None and top10.index[1] == 'SIG_104')
check('1위 효과크기 0.627', top10 is not None and abs(top10.iloc[0] - 0.6265) < 0.01)

---

## 미션 7 · 시간에 따른 불량률 변화

장비는 시간이 지나면서 상태가 변합니다. 소모품이 닳고, 챔버가 오염되고, 정기점검을 받습니다.
예지보전의 출발점은 불량이 언제 몰렸는지 보는 것입니다.

월별로 처리량과 불량률을 집계하고 선그래프로 그리세요.
그래프를 보면서 미션 8에서 뭐라고 쓸지 생각해 두세요.

In [ ]:
# TODO 7-1: 월 단위 컬럼 만들기
#   힌트: df['timestamp'].dt.to_period('M')
#   (아래에 직접 작성하세요)


# TODO 7-2: 월별 처리량 / 불량수 / 불량률
monthly = None

# TODO 7-3: 월별 불량률 선그래프


자가진단 7 — 실행해서 모두 `[통과]` 인지 확인하세요.

In [ ]:
check('month 컬럼', df is not None and 'month' in df.columns and df['month'].notna().all())
check('4개월 집계', monthly is not None and len(monthly) == 4, '2008년 7~10월')
check('7월이 가장 높음', monthly is not None and monthly['불량률'].idxmax() == monthly.index[0])

---

## 미션 8 · 정리해서 쓰기

여기부터는 코드가 아니라 글입니다. 아래 항목을 본인 문장으로 채우세요.
숫자를 근거로 인용해야 합니다. 446개, 6.64% 같은 식으로요.

---

**1. 데이터가 어떤 것이었나**

웨이퍼 몇 장, 신호 몇 개, 불량률은 얼마였습니까.

답: 
웨이퍼 1,567장에 대해 590개의 신호가 기록되어있습니다. 이 중 불량 판정이 104건이라 불량률은 6.64%였습니다.

**2. 무엇을 왜 버렸나**

590개 중 144개를 뺐습니다. 어떤 기준으로 뺐고, 그 기준이 왜 타당합니까.

답: 
실제 장비 데이터에는 비어 있는 값이 많고, 센서가 고장 났거나, 그 레시피에서는 아예 측정하지 않는 항목이거나, 통신이 끊긴 경우 데이터가 비었다고 했습니다. 그럼 비어있는 값을 한번 찾아봐야하는데, 50% 기준으로 그 이상의 빈 데이터는 제가 평균값으로 채워넣어도 의미없는 데이터라고 판단, 버리고, 값이 처음부터 끝까지 똑같은 신호라면 아무리 양품이라고 해도 확률은 극히낮고 불량일 확률이 압도적으로 높다고 판단하여 그 값들을 버렸습니다.

**3. 어떤 신호가 걸렸나**

Top 10에 어느 모듈이 많이 나왔습니까. 1등 신호의 박스플롯에서 무엇이 보였습니까.

답: 
모듈의 분포 구성은 가스공급2개, 계측2개, 이송2개, 온도제어, 정전척, 진공 등 고르게 분포되어서 어느 모듈이 많이 나왔다 답은 불가합니다.

1등 신호 (SIG_060)의 박스 플롯에선 1개만 얻을 수 있었습니다. "신호값으로 양품과 불량을 구별하는건 힘들것 같다"입니다.
3가지 근거가 있습니다.
1. 양품과 불량의 효과크기의 차이는 있었습니다. 불량쪽의 그래프가 양수 방향으로 높이가 더 높았습니다. 
2. 양품과 불량의 효과크기는 집약의 차이만 있을뿐 둘다 0 주변에 위치해 있습니다.
3. 양품 쪽에는 분포가 다양했습니다. 음수값도 존재하고 최대 168.15까지 튀는 수치가 있습니다. 오히려 불량보다 신호값 범위가 더 넓다 볼 수 있습니다.
사료값이 부족해 판단을 내리기엔 미흡하다 생각하지만, 그걸 감안하더라도 신호값의 높낮이를 근거로 양품과 불량을 판단하기 어려울것 같습니다.

**4. 시간에 따라 어떻게 변했나**

월별 불량률이 어떻게 움직였고, 원인으로 무엇을 의심할 수 있습니까.

답: 7월이 22.22%로 압도적으로 높았고 8월 9.19%, 9월 2.88%까지 내려갔다가 10월에 6.13%로 다시 올라갔습니다. 전체 평균이 6.64%니까 7월은 세 배가 넘습니다.
하지만 7월의 처리량이 63장입니다. 이정도 수치면, 7월에 새로이 들여온 장비가 가동시작을 했지만, 원하는 공정과 새로운 환경과 맞물려서 적절한 공정 값을 세팅하기 위해 장비를 가동한것이 목적인것 같고, 불량률이 굉장히 높은 로그가 찍힌 이유가 될 것 같습니다. 
8월은 공정을 돌려면서 개선을 진행시켜 9월엔 안정화가 되었고, 10월에는 장비가 갖고있는 소모품의 수명 문제라던지 장비에 공급되는 재료의 비율이 달라지는등의 요소 때문에 이와 같은 반응이 일어난거 같습니다. 다만 미션 7의 문제에서 소모품이 닳고 챔버가 오염되며 정기점검을 받아야할 시간이라고 언급하셨기에 이것이 이유가 될거 같습니다. 실무 경험이 없어서 추측하기 어려운것 같습니다.

**5. 공정팀에 보내는 한 문단**

어느 모듈을 먼저 점검하라고 권하겠습니까. 근거를 넣어 한 문단으로 쓰세요.

답: 수치의 신뢰도가 높다는 가정하에, 가스공급 모듈의 라인압력을 확인해주세요. 446개의 신호중 효과크기 1위를 차지하는 모듈이 가스공급 모듈이고 9위에 한번더 위치해 있기 때문에 우선하는게 좋을거 같습니다. 라고 권할거 같습니다.

**6. 이 분석으로 말할 수 없는 것**

이 결과만으로 단정할 수 없는 이유를 최소 두 가지 쓰세요.
효과크기가 크면 그 신호가 원인입니까. 불량 104건은 충분합니까. 신호 이름은 진짜입니까.

답: 
상위 신호에 위치한 가스공급 모듈의 신호를 보면 양품 쪽에는 음수값부터 최대 168.15까지 튀는 수치를 보면 분포가 다양했습니다. 불량보다 신호값 범위가 더 넓다 볼 수 있습니다. 적은 불량 표본과 양/불 신호의 분포 차이를 무시하는 효과크기는 단독으로 사용한다면 신뢰도가 부족하다 생각합니다. 
특히 현재 데이터의 불량 건수가 104건에 불과하고, 월별 데이터의 분포도 고르지 않기 때문에 이를 이용한 분석은 불량을 잡아낼수있는 수치를 의미하는 것인지 판단하기엔 표본이 부족합니다. 
7월은 63개의 데이터만 존재하면서 불량률이 22.22%로 매우 높고, 장비의 초기 세팅이나 테스트가 이루어진 시기일 가능성도 있어 일반적인 공정 상태로 보기 어렵고, 10월 역시 데이터가 완전하지 않다는 점에서 4개월의 데이터를 동일한 조건으로 비교하기 힘듭니다. 

이번 분석에서 효과크기가 큰 신호가 발견되었다고 해서 그 신호가 불량의 원인이라고 단정할 수는 없습니다. 효과크기는 양품과 불량 사이에서 신호값의 차이가 얼마나 크게 나타나는지를 보여주는 지표일 뿐, 해당 신호가 불량을 직접 발생시켰다는 인과관계까지 보여주지는 않기 때문입니다 
따라서 효과크기가 큰 신호는 장비의 점검 범위를 좁히는 데 참고할 수 있는 지표이지만, 절대적인 원인 판단 기준으로 사용하기는 어려운것 같습니다. 
실제 원인이라고 판단하려면 장비 로그나 공정 조건 등 추가적인 근거가 필요할것 같습니다. 

그리고 메타데이터에 써있고 수업시간에 교수님이 언급 하셨던 말씀을 생각해보면 모든 신호가 장비의 이름을 태그로 달고 날라오는것이 아닌, 익명의 신호로 전달이 된다고 했습니다. 그렇다면 실제 펩에선 이 신호가 어디서 왔는지 분류가 불가 할것이고, 이런식의 분석이 애초에 불가합니다. 다른 방법이 필요할것 같습니다.

---

### 제출 전 확인

- [ ] 자가진단이 모두 `[통과]` 인가
- [ ] 그래프 세 개(막대·박스·선)에 제목과 축 이름이 있는가
- [ ] 미션 8의 여섯 항목을 본인 문장으로 채웠는가
- [ ] 커널 재시작 후 전체 실행이 오류 없이 끝나는가
- [ ] 파일명을 `W01_학번_이름.ipynb` 로 바꿨는가